In [53]:
# ============================================================
# NewsGuard — Real-World Generalization Diagnosis
# Cell 1: Restore & Verify
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import joblib

from sklearn.feature_selection import SelectKBest, chi2

BASE_DIR = "/content/drive/MyDrive/NewsGuard"

# Restore model + TF-IDF
tfidf_vectorizer = joblib.load(
    os.path.join(
        BASE_DIR,
        "features",
        "tfidf",
        "tfidf_vectorizer.joblib"
    )
)

best_xgb = joblib.load(
    os.path.join(
        BASE_DIR,
        "results",
        "day_06",
        "day_06_xgboost_best_tuned.joblib"
    )
)

# Restore training TF-IDF + labels
X_train_tfidf = sparse.load_npz(
    os.path.join(
        BASE_DIR,
        "features",
        "tfidf",
        "X_train_tfidf.npz"
    )
)

y_train = pd.read_csv(
    os.path.join(
        BASE_DIR,
        "data",
        "splits",
        "train.csv"
    )
)["label"].values

# Recreate exact Day 6/7 selector
selector_diag = SelectKBest(
    score_func=chi2,
    k=500
)

selector_diag.fit(
    X_train_tfidf,
    y_train
)

print("TF-IDF features:", tfidf_vectorizer.max_features)
print("Selected TF-IDF:", selector_diag.k)
print("Model features:", best_xgb.n_features_in_)
print("Train shape:", X_train_tfidf.shape)

assert best_xgb.n_features_in_ == 615

print("\n✅ DIAGNOSTIC RESTORE PASSED")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TF-IDF features: 5000
Selected TF-IDF: 500
Model features: 615
Train shape: (31282, 5000)

✅ DIAGNOSTIC RESTORE PASSED


In [54]:
# ============================================================
# Cell 2: Diagnose Reuters Prediction
# ============================================================

reuters_test = """
India's foreign exchange reserves climbed to a record high
of $785.7 billion in the week to September 4, central bank
data showed on Friday, lifted by a wave of inflows under
policy measures to strengthen the country's balance of payments.

India's FX reserves have risen for ten straight weeks,
jumping by almost $120 billion during this period. The latest
data showed a nearly $45 billion week-on-week rise.

The rise in reserves was led by a $47.4 billion gain in the
central bank's foreign currency assets, while the value of
its gold holdings declined by about $2.6 billion.

The Reserve Bank of India has been intervening in the foreign
exchange market to support the rupee.
"""

# Prepare exact 615 features
cleaned = clean_text(reuters_test)

tfidf = tfidf_vectorizer.transform([cleaned])
selected = selector_diag.transform(tfidf)

aux = extract_exact_auxiliary_features(reuters_test)

aux_sparse = sparse.csr_matrix(
    aux,
    dtype=np.float32
)

X_reuters = sparse.hstack(
    [selected, aux_sparse],
    format="csr"
)

print("Feature shape:", X_reuters.shape)

# Prediction
probs = best_xgb.predict_proba(X_reuters)[0]

print("\nPrediction:", "Real" if np.argmax(probs) == 1 else "Fake")
print("Fake probability:", round(float(probs[0]), 6))
print("Real probability:", round(float(probs[1]), 6))

# SHAP
import shap

explainer = shap.TreeExplainer(best_xgb)

X_dense = X_reuters.toarray()

shap_values = explainer.shap_values(X_dense)

if isinstance(shap_values, list):
    shap_row = shap_values[1][0]
else:
    shap_row = shap_values[0]

# Feature names
tfidf_names = np.array(
    tfidf_vectorizer.get_feature_names_out()
)[selector_diag.get_support()]

feature_names = np.concatenate([
    tfidf_names,
    np.array(AUXILIARY_FEATURE_NAMES)
])

# Top positive/negative contributions
top_idx = np.argsort(np.abs(shap_row))[::-1][:30]

diagnosis = pd.DataFrame({
    "feature": feature_names[top_idx],
    "shap_value": shap_row[top_idx]
})

print("\nTop 30 SHAP features:")
display(diagnosis)

Feature shape: (1, 615)

Prediction: Fake
Fake probability: 0.997968
Real probability: 0.002032

Top 30 SHAP features:


,feature,shap_value
0,read more,-1.505592
1,reuters,1.394744
2,washington reuters,1.034901
3,featured image,-0.993013
4,century wire,-0.968295
5,nov,0.884114
6,said,-0.752351
7,getty,-0.419470
8,Character Count,-0.294749
9,W2V_070,-0.283163


In [55]:
# ============================================================
# Cell 3 — Feature Ablation: Suspicious TF-IDF Terms
# ============================================================

suspicious_terms = [
    "read more",
    "reuters",
    "washington reuters",
    "featured image",
    "century wire",
    "nov",
    "said",
    "getty",
    "getty images",
    "monday",
    "via",
    "the nov",
    "president barack",
    "it said"
]

# Check which suspicious terms are actually selected
selected_feature_set = set(tfidf_names)

present_terms = [
    term for term in suspicious_terms
    if term in selected_feature_set
]

print("Suspicious selected features:")
for term in present_terms:
    print("→", term)

print("\nCount:", len(present_terms))

Suspicious selected features:
→ read more
→ reuters
→ washington reuters
→ featured image
→ century wire
→ nov
→ said
→ getty
→ getty images
→ monday
→ via
→ the nov
→ president barack
→ it said

Count: 14


In [56]:
# ============================================================
# Cell 4 — Ablation Test
# ============================================================

X_ablation = X_reuters.copy().toarray()

# Remove suspicious TF-IDF feature values
for term in present_terms:

    feature_idx = np.where(
        feature_names == term
    )[0]

    if len(feature_idx) > 0:
        X_ablation[0, feature_idx[0]] = 0.0

ablation_probs = best_xgb.predict_proba(
    X_ablation
)[0]

ablation_prediction = (
    "Real"
    if np.argmax(ablation_probs) == 1
    else "Fake"
)

print("Original:")
print("Prediction:", "Fake")
print("Fake probability:", round(float(probs[0]), 6))
print("Real probability:", round(float(probs[1]), 6))

print("\nAfter removing suspicious TF-IDF features:")
print("Prediction:", ablation_prediction)
print("Fake probability:", round(float(ablation_probs[0]), 6))
print("Real probability:", round(float(ablation_probs[1]), 6))
print("Credibility:", round(float(ablation_probs[1] * 100), 2))

Original:
Prediction: Fake
Fake probability: 0.997968
Real probability: 0.002032

After removing suspicious TF-IDF features:
Prediction: Fake
Fake probability: 0.777871
Real probability: 0.222129
Credibility: 22.21


In [57]:
# ============================================================
# Cell 5 — TF-IDF vs Auxiliary Feature Ablation
# ============================================================

X_original = X_reuters.toarray()

# -------------------------------
# A) TF-IDF ONLY
# -------------------------------
X_tfidf_only = X_original.copy()

# Remove all 115 auxiliary features
X_tfidf_only[:, 500:] = 0.0

tfidf_probs = best_xgb.predict_proba(
    X_tfidf_only
)[0]


# -------------------------------
# B) AUXILIARY ONLY
# -------------------------------
X_aux_only = X_original.copy()

# Remove all 500 TF-IDF features
X_aux_only[:, :500] = 0.0

aux_probs = best_xgb.predict_proba(
    X_aux_only
)[0]


print("TF-IDF ONLY")
print("Fake:", round(float(tfidf_probs[0]), 6))
print("Real:", round(float(tfidf_probs[1]), 6))
print("Credibility:", round(float(tfidf_probs[1] * 100), 2))

print("\nAUXILIARY ONLY")
print("Fake:", round(float(aux_probs[0]), 6))
print("Real:", round(float(aux_probs[1]), 6))
print("Credibility:", round(float(aux_probs[1] * 100), 2))

TF-IDF ONLY
Fake: 0.756408
Real: 0.243592
Credibility: 24.36

AUXILIARY ONLY
Fake: 0.789041
Real: 0.210959
Credibility: 21.1


In [58]:
# ============================================================
# Cell 6 — Complete Ablation Comparison
# ============================================================

comparison = pd.DataFrame([
    {
        "Experiment": "Original 615 Features",
        "Fake Probability": probs[0],
        "Real Probability": probs[1],
        "Credibility": probs[1] * 100
    },
    {
        "Experiment": "TF-IDF Only",
        "Fake Probability": tfidf_probs[0],
        "Real Probability": tfidf_probs[1],
        "Credibility": tfidf_probs[1] * 100
    },
    {
        "Experiment": "Auxiliary Only",
        "Fake Probability": aux_probs[0],
        "Real Probability": aux_probs[1],
        "Credibility": aux_probs[1] * 100
    },
    {
        "Experiment": "Suspicious TF-IDF Removed",
        "Fake Probability": ablation_probs[0],
        "Real Probability": ablation_probs[1],
        "Credibility": ablation_probs[1] * 100
    }
])

display(
    comparison.style.format({
        "Fake Probability": "{:.4f}",
        "Real Probability": "{:.4f}",
        "Credibility": "{:.2f}"
    })
)

,Experiment,Fake Probability,Real Probability,Credibility
0,Original 615 Features,0.9980,0.0020,0.20
1,TF-IDF Only,0.7564,0.2436,24.36
2,Auxiliary Only,0.7890,0.2110,21.10
3,Suspicious TF-IDF Removed,0.7779,0.2221,22.21


In [59]:
# ============================================================
# Cell 7 — Training Feature Bias Diagnosis
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

train_df = pd.read_csv(
    os.path.join(
        BASE_DIR,
        "data",
        "splits",
        "train.csv"
    )
)

fake_texts = train_df.loc[
    train_df["label"] == 0,
    "content"
].fillna("").values

real_texts = train_df.loc[
    train_df["label"] == 1,
    "content"
].fillna("").values

bias_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True
)

X_fake = bias_vectorizer.fit_transform(fake_texts)
X_real = bias_vectorizer.transform(real_texts)

feature_names_bias = np.array(
    bias_vectorizer.get_feature_names_out()
)

fake_mean = np.asarray(X_fake.mean(axis=0)).ravel()
real_mean = np.asarray(X_real.mean(axis=0)).ravel()

bias_score = real_mean - fake_mean

top_real_idx = np.argsort(bias_score)[-30:][::-1]
top_fake_idx = np.argsort(bias_score)[:30]

print("TOP FEATURES ASSOCIATED WITH REAL")
display(pd.DataFrame({
    "feature": feature_names_bias[top_real_idx],
    "real_mean_tfidf": real_mean[top_real_idx],
    "fake_mean_tfidf": fake_mean[top_real_idx],
    "bias_score": bias_score[top_real_idx]
}))

print("\nTOP FEATURES ASSOCIATED WITH FAKE")
display(pd.DataFrame({
    "feature": feature_names_bias[top_fake_idx],
    "real_mean_tfidf": real_mean[top_fake_idx],
    "fake_mean_tfidf": fake_mean[top_fake_idx],
    "bias_score": bias_score[top_fake_idx]
}))

TOP FEATURES ASSOCIATED WITH REAL


,feature,real_mean_tfidf,fake_mean_tfidf,bias_score
0,reuters,0.098605,0.000907,0.097698
1,said on,0.035906,0.001161,0.034745
2,said,0.047792,0.017422,0.030370
3,minister,0.022982,0.001760,0.021222
4,washington,0.022901,0.005743,0.017159
5,its,0.024047,0.007608,0.016439
6,said the,0.020357,0.004077,0.016280
7,on wednesday,0.018299,0.002126,0.016174
8,said in,0.018895,0.002964,0.015931
9,president donald,0.017269,0.002002,0.015267



TOP FEATURES ASSOCIATED WITH FAKE


,feature,real_mean_tfidf,fake_mean_tfidf,bias_score
0,video,0.000797,0.018738,-0.017941
1,you,0.003641,0.020347,-0.016706
2,just,0.003510,0.015324,-0.011815
3,this,0.011482,0.023045,-0.011563
4,via,0.000466,0.011610,-0.011144
5,hillary,0.003495,0.013899,-0.010403
6,image,0.000289,0.010600,-0.010311
7,watch,0.000625,0.010612,-0.009988
8,what,0.006133,0.016003,-0.009870
9,like,0.003877,0.013732,-0.009855


In [60]:
# ============================================================
# Cell 8 — Explicit Source/Style Bias Check
# ============================================================

check_terms = [
    "reuters",
    "washington reuters",
    "read more",
    "featured image",
    "century wire",
    "getty",
    "getty images",
    "via",
    "said",
    "it said",
    "nov",
    "monday"
]

rows = []

for term in check_terms:

    fake_mask = train_df["label"] == 0
    real_mask = train_df["label"] == 1

    fake_contains = train_df.loc[
        fake_mask,
        "content"
    ].str.lower().str.contains(
        term,
        regex=False,
        na=False
    ).mean()

    real_contains = train_df.loc[
        real_mask,
        "content"
    ].str.lower().str.contains(
        term,
        regex=False,
        na=False
    ).mean()

    rows.append({
        "term": term,
        "fake_percentage": fake_contains * 100,
        "real_percentage": real_contains * 100,
        "real_minus_fake": (
            real_contains - fake_contains
        ) * 100
    })

source_bias = pd.DataFrame(rows)

display(
    source_bias.sort_values(
        "real_minus_fake",
        ascending=False
    ).style.format({
        "fake_percentage": "{:.2f}%",
        "real_percentage": "{:.2f}%",
        "real_minus_fake": "{:+.2f}%"
    })
)

,term,fake_percentage,real_percentage,real_minus_fake
0,reuters,1.21%,99.81%,+98.60%
8,said,55.57%,93.81%,+38.24%
11,monday,7.93%,22.95%,+15.02%
10,nov,10.02%,17.90%,+7.88%
9,it said,0.38%,6.27%,+5.89%
1,washington reuters,0.00%,0.00%,+0.00%
4,century wire,3.55%,0.00%,-3.55%
2,read more,9.51%,0.05%,-9.47%
6,getty images,22.06%,0.00%,-22.06%
5,getty,22.70%,0.02%,-22.68%


In [61]:
# ============================================================
# Cell 9 — Source / Template Artifact Removal
# ============================================================

import re

SOURCE_ARTIFACT_PATTERNS = [
    r"\breuters\b",
    r"\bwashington reuters\b",
    r"\bgetty images\b",
    r"\bgetty\b",
    r"\bfeatured image\b",
    r"\bread more\b",
    r"\bcentury wire\b",
    r"\bpic twitter\b",
    r"\btwitter com\b",
    r"\bimage via\b",
    r"\bvia\b"
]

def remove_source_artifacts(text):

    text = str(text).lower()

    for pattern in SOURCE_ARTIFACT_PATTERNS:
        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(r"\s+", " ", text).strip()

    return text


# Test on Reuters article
cleaned_reuters_artifact = remove_source_artifacts(
    reuters_test
)

print("Original length:", len(reuters_test))
print("Cleaned length:", len(cleaned_reuters_artifact))

print("\nCleaned Reuters sample:")
print(cleaned_reuters_artifact[:1000])

print("\n✅ SOURCE ARTIFACT REMOVAL READY")

Original length: 682
Cleaned length: 677

Cleaned Reuters sample:
india's foreign exchange reserves climbed to a record high of $785.7 billion in the week to september 4, central bank data showed on friday, lifted by a wave of inflows under policy measures to strengthen the country's balance of payments. india's fx reserves have risen for ten straight weeks, jumping by almost $120 billion during this period. the latest data showed a nearly $45 billion week-on-week rise. the rise in reserves was led by a $47.4 billion gain in the central bank's foreign currency assets, while the value of its gold holdings declined by about $2.6 billion. the reserve bank of india has been intervening in the foreign exchange market to support the rupee.

✅ SOURCE ARTIFACT REMOVAL READY


In [62]:
# ============================================================
# Cell 10 — Create Source-Agnostic Training Text
# ============================================================

train_source_agnostic = train_df.copy()

train_source_agnostic["content_source_agnostic"] = (
    train_source_agnostic["content"]
    .fillna("")
    .apply(remove_source_artifacts)
)

print("Original training rows:",
      len(train_source_agnostic))

print(
    "Empty after artifact removal:",
    train_source_agnostic[
        "content_source_agnostic"
    ].str.strip().eq("").sum()
)

print("\nExample:")
print(
    train_source_agnostic[
        ["content", "content_source_agnostic"]
    ].head(2).to_string(index=False)
)

print("\n✅ SOURCE-AGNOSTIC TRAINING TEXT CREATED")

Original training rows: 31282
Empty after artifact removal: 0

Example:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [63]:
# ============================================================
# Cell 11 — Check Cleaned Reuters TF-IDF Representation
# ============================================================

reuters_cleaned = remove_source_artifacts(
    reuters_test
)

reuters_cleaned_tfidf = tfidf_vectorizer.transform(
    [reuters_cleaned]
)

selected_cleaned = selector_diag.transform(
    reuters_cleaned_tfidf
)

print("Original TF-IDF non-zero:",
      X_reuters[:, :500].count_nonzero())

print("Cleaned TF-IDF non-zero:",
      selected_cleaned.count_nonzero())

remaining_artifacts = [
    term for term in SOURCE_ARTIFACT_PATTERNS
    if re.search(
        term,
        reuters_cleaned,
        flags=re.IGNORECASE
    )
]

print("\nRemaining source artifacts:")
print(remaining_artifacts)

print("\n✅ CLEANED INPUT VERIFIED")

Original TF-IDF non-zero: 11
Cleaned TF-IDF non-zero: 11

Remaining source artifacts:
[]

✅ CLEANED INPUT VERIFIED


In [64]:
# ============================================================
# Cell 12 — Source-Agnostic TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

source_agnostic_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype="float32"
)

X_train_source_tfidf = source_agnostic_vectorizer.fit_transform(
    train_source_agnostic["content_source_agnostic"]
)

print("Source-agnostic TF-IDF shape:",
      X_train_source_tfidf.shape)

print("Vocabulary size:",
      len(source_agnostic_vectorizer.vocabulary_))

# Verify major source artifacts are gone from vocabulary
source_terms_in_vocab = [
    term for term in [
        "reuters",
        "washington reuters",
        "getty",
        "getty images",
        "featured image",
        "read more",
        "century wire"
    ]
    if term in source_agnostic_vectorizer.vocabulary_
]

print("\nSource artifacts still in vocabulary:")
print(source_terms_in_vocab)

print("\n✅ SOURCE-AGNOSTIC TF-IDF CREATED")

/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Source-agnostic TF-IDF shape: (31282, 5000)
Vocabulary size: 5000

Source artifacts still in vocabulary:
[]

✅ SOURCE-AGNOSTIC TF-IDF CREATED


In [65]:
# ============================================================
# Cell 13 — Source-Agnostic 615-Feature Dataset
# ============================================================

source_selector = SelectKBest(
    score_func=chi2,
    k=500
)

X_train_source_selected = source_selector.fit_transform(
    X_train_source_tfidf,
    y_train
)

# Reuse the EXACT Day 4 auxiliary features
X_train_aux_source = np.load(
    os.path.join(
        BASE_DIR,
        "features",
        "auxiliary",
        "X_train_auxiliary.npy"
    )
).astype(np.float32)

X_train_source_combined = sparse.hstack(
    [
        X_train_source_selected,
        sparse.csr_matrix(X_train_aux_source)
    ],
    format="csr"
)

print("Selected TF-IDF:",
      X_train_source_selected.shape)

print("Auxiliary:",
      X_train_aux_source.shape)

print("Final source-agnostic features:",
      X_train_source_combined.shape)

assert X_train_source_combined.shape[1] == 615

print("\n✅ SOURCE-AGNOSTIC 615-FEATURE DATASET READY")

Selected TF-IDF: (31282, 500)
Auxiliary: (31282, 115)
Final source-agnostic features: (31282, 615)

✅ SOURCE-AGNOSTIC 615-FEATURE DATASET READY


In [66]:
# ============================================================
# Cell 14 — Experimental Source-Agnostic XGBoost
# ============================================================

from xgboost import XGBClassifier

source_agnostic_xgb = XGBClassifier(
    n_estimators=120,
    max_depth=3,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

source_agnostic_xgb.fit(
    X_train_source_combined,
    y_train
)

print("✅ SOURCE-AGNOSTIC XGBOOST TRAINED")
print("Features:", source_agnostic_xgb.n_features_in_)

✅ SOURCE-AGNOSTIC XGBOOST TRAINED
Features: 615


In [67]:
# ============================================================
# Cell 15 — Reuters Test: Source-Agnostic Model
# ============================================================

# Clean Reuters article using the SAME source-artifact removal
reuters_source_clean = remove_source_artifacts(reuters_test)

# TF-IDF
X_reuters_source_tfidf = (
    source_agnostic_vectorizer.transform(
        [reuters_source_clean]
    )
)

# Select same 500 features
X_reuters_source_selected = (
    source_selector.transform(
        X_reuters_source_tfidf
    )
)

# Exact 115 auxiliary features
reuters_aux = extract_exact_auxiliary_features(
    reuters_test
)

X_reuters_source_aux = sparse.csr_matrix(
    reuters_aux,
    dtype=np.float32
)

# Final 615 features
X_reuters_source = sparse.hstack(
    [
        X_reuters_source_selected,
        X_reuters_source_aux
    ],
    format="csr"
)

# Prediction
source_probs = source_agnostic_xgb.predict_proba(
    X_reuters_source
)[0]

source_prediction = (
    "Real"
    if np.argmax(source_probs) == 1
    else "Fake"
)

print("SOURCE-AGNOSTIC MODEL")
print("=" * 45)
print("Prediction:", source_prediction)
print("Fake Probability:", round(float(source_probs[0]), 6))
print("Real Probability:", round(float(source_probs[1]), 6))
print("Credibility Score:", round(float(source_probs[1] * 100), 2))

SOURCE-AGNOSTIC MODEL
Prediction: Fake
Fake Probability: 0.977628
Real Probability: 0.022372
Credibility Score: 2.24


In [68]:
# ============================================================
# Cell 16 — Old vs Source-Agnostic Model
# ============================================================

model_comparison = pd.DataFrame([
    {
        "Model": "Original NewsGuard",
        "Prediction": "Fake",
        "Fake Probability": float(probs[0]),
        "Real Probability": float(probs[1]),
        "Credibility": float(probs[1] * 100)
    },
    {
        "Model": "Source-Agnostic Experimental",
        "Prediction": source_prediction,
        "Fake Probability": float(source_probs[0]),
        "Real Probability": float(source_probs[1]),
        "Credibility": float(source_probs[1] * 100)
    }
])

display(
    model_comparison.style.format({
        "Fake Probability": "{:.4f}",
        "Real Probability": "{:.4f}",
        "Credibility": "{:.2f}"
    })
)

,Model,Prediction,Fake Probability,Real Probability,Credibility
0,Original NewsGuard,Fake,0.9980,0.0020,0.20
1,Source-Agnostic Experimental,Fake,0.9776,0.0224,2.24


In [69]:
# ============================================================
# Cell 17 — Source-Agnostic Held-Out Test Set
# ============================================================

test_df = pd.read_csv(
    os.path.join(
        BASE_DIR,
        "data",
        "splits",
        "test.csv"
    )
)

y_test_source = test_df["label"].values

test_df["content_source_agnostic"] = (
    test_df["content"]
    .fillna("")
    .apply(remove_source_artifacts)
)

X_test_source_tfidf = (
    source_agnostic_vectorizer.transform(
        test_df["content_source_agnostic"]
    )
)

X_test_source_selected = (
    source_selector.transform(
        X_test_source_tfidf
    )
)

X_test_aux_source = np.load(
    os.path.join(
        BASE_DIR,
        "features",
        "auxiliary",
        "X_test_auxiliary.npy"
    )
).astype(np.float32)

X_test_source_combined = sparse.hstack(
    [
        X_test_source_selected,
        sparse.csr_matrix(X_test_aux_source)
    ],
    format="csr"
)

print("Test TF-IDF:", X_test_source_selected.shape)
print("Test auxiliary:", X_test_aux_source.shape)
print("Final test:", X_test_source_combined.shape)

assert X_test_source_combined.shape == (len(test_df), 615)

print("\n✅ SOURCE-AGNOSTIC TEST SET READY")

Test TF-IDF: (3911, 500)
Test auxiliary: (3911, 115)
Final test: (3911, 615)

✅ SOURCE-AGNOSTIC TEST SET READY


In [70]:
# ============================================================
# Cell 18 — Source-Agnostic Model Evaluation
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_test_pred_source = (
    source_agnostic_xgb.predict(
        X_test_source_combined
    )
)

y_test_prob_source = (
    source_agnostic_xgb.predict_proba(
        X_test_source_combined
    )[:, 1]
)

source_accuracy = accuracy_score(
    y_test_source,
    y_test_pred_source
)

source_precision = precision_score(
    y_test_source,
    y_test_pred_source
)

source_recall = recall_score(
    y_test_source,
    y_test_pred_source
)

source_f1 = f1_score(
    y_test_source,
    y_test_pred_source
)

source_auc = roc_auc_score(
    y_test_source,
    y_test_prob_source
)

print("SOURCE-AGNOSTIC TEST PERFORMANCE")
print("=" * 50)
print("Accuracy :", round(source_accuracy, 6))
print("Precision:", round(source_precision, 6))
print("Recall   :", round(source_recall, 6))
print("F1       :", round(source_f1, 6))
print("ROC-AUC  :", round(source_auc, 6))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_source,
        y_test_pred_source
    )
)

SOURCE-AGNOSTIC TEST PERFORMANCE
Accuracy : 0.987727
Precision: 0.9873
Recall   : 0.990094
F1       : 0.988695
ROC-AUC  : 0.999328

Confusion Matrix:
[[1764   27]
 [  21 2099]]


In [71]:
# ============================================================
# Cell 19 — Original vs Source-Agnostic
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "Original Day 7",
        "Accuracy": 0.997187,
        "F1": 0.997410,
        "ROC_AUC": 0.999780
    },
    {
        "Model": "Source-Agnostic",
        "Accuracy": source_accuracy,
        "F1": source_f1,
        "ROC_AUC": source_auc
    }
])

display(
    comparison.style.format({
        "Accuracy": "{:.4f}",
        "F1": "{:.4f}",
        "ROC_AUC": "{:.4f}"
    })
)

,Model,Accuracy,F1,ROC_AUC
0,Original Day 7,0.9972,0.9974,0.9998
1,Source-Agnostic,0.9877,0.9887,0.9993


In [72]:
# ============================================================
# CELL 20 — EXTERNAL VALIDATION SETUP (WELFake)
# IMPORTANT: Does NOT modify Day 1–8 artifacts
# ============================================================

!pip install -q datasets

from datasets import load_dataset
import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

EXTERNAL_DIR = "/content/drive/MyDrive/NewsGuard/experiments/external_validation"
os.makedirs(EXTERNAL_DIR, exist_ok=True)

print("Loading WELFake dataset...")

welfake = load_dataset("davanstrien/WELFake")

print(welfake)
print("\nColumns:", welfake["train"].column_names)
print("Rows:", len(welfake["train"]))

print("\nLabel distribution:")
print(welfake["train"].to_pandas()["label"].value_counts().sort_index())

print("\n✅ WELFAKE EXTERNAL DATASET LOADED")

Loading WELFake dataset...


README.md:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

data/train-00000-of-00001-290868f0a36350(…): reconstructing file:   0%|          |  0.00B /  152MB            

data/train-00000-of-00001-290868f0a36350(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/72134 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['title', 'text', 'label'],
        num_rows: 72134
    })
})

Columns: ['title', 'text', 'label']
Rows: 72134

Label distribution:
label
0    35028
1    37106
Name: count, dtype: int64

✅ WELFAKE EXTERNAL DATASET LOADED


In [73]:
# ============================================================
# CELL 21 — CREATE BALANCED EXTERNAL VALIDATION SAMPLE
# ============================================================

welfake_df = welfake["train"].to_pandas()

# Keep only required columns
welfake_df = welfake_df[["title", "text", "label"]].copy()

# Remove missing/empty articles
welfake_df["title"] = welfake_df["title"].fillna("")
welfake_df["text"] = welfake_df["text"].fillna("")

welfake_df["content"] = (
    welfake_df["title"].astype(str)
    + " "
    + welfake_df["text"].astype(str)
).str.strip()

welfake_df = welfake_df[
    welfake_df["content"].str.len() > 50
].copy()

# Balanced sample: 1000 fake + 1000 real
fake_df = welfake_df[welfake_df["label"] == 0].sample(
    n=1000,
    random_state=42
)

real_df = welfake_df[welfake_df["label"] == 1].sample(
    n=1000,
    random_state=42
)

external_df = pd.concat(
    [fake_df, real_df],
    ignore_index=True
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("External sample shape:", external_df.shape)
print("\nLabel distribution:")
print(external_df["label"].value_counts())

print("\nSample article:")
print(external_df.iloc[0]["content"][:500])

# Save separately
external_df.to_csv(
    os.path.join(EXTERNAL_DIR, "welfake_external_sample_2000.csv"),
    index=False
)

print("\n✅ EXTERNAL VALIDATION SAMPLE SAVED")

External sample shape: (2000, 4)

Label distribution:
label
1    1000
0    1000
Name: count, dtype: int64

Sample article:
Collective Consciousness: The Individual is Gone Print Email http://humansarefree.com/2016/11/collective-consciousness-individual-is.html In the middle of all the brain-research going on, from one end of the planet to the other, there is the assumption that the individual doesn’t really exist. He’s a fiction. There is only the motion of particles in the brain. Therefore, nothing is inviolate, nothing is protected. Make the brain do A, make it do B; it doesn’t matter. What matters is harmonizing 

✅ EXTERNAL VALIDATION SAMPLE SAVED


In [75]:
# ============================================================
# CELL 22 — RESTORE ORIGINAL DAY 7 MODEL
# IMPORTANT: DAY 1–8 ARTIFACTS ARE READ-ONLY
# ============================================================

import joblib
from scipy.sparse import load_npz
from sklearn.feature_selection import SelectKBest, chi2

BASE_DIR = "/content/drive/MyDrive/NewsGuard"

tfidf_vectorizer = joblib.load(
    f"{BASE_DIR}/features/tfidf/tfidf_vectorizer.joblib"
)

best_xgb = joblib.load(
    f"{BASE_DIR}/results/day_06/day_06_xgboost_best_tuned.joblib"
)

auxiliary_scaler = joblib.load(
    f"{BASE_DIR}/features/combined/auxiliary_scaler.joblib"
)

# Restore ORIGINAL training TF-IDF
X_train_tfidf_original = load_npz(
    f"{BASE_DIR}/features/tfidf/X_train_tfidf.npz"
)

train_df_original = pd.read_csv(
    f"{BASE_DIR}/data/splits/train.csv"
)

y_train_original = train_df_original["label"].values

# Recreate the EXACT Day 7 selector
source_selector = SelectKBest(
    score_func=chi2,
    k=500
)

source_selector.fit(
    X_train_tfidf_original,
    y_train_original
)

print("Original TF-IDF:", X_train_tfidf_original.shape)
print("Selected TF-IDF:", source_selector.k)
print("Model features:", best_xgb.n_features_in_)

assert X_train_tfidf_original.shape[1] == 5000
assert source_selector.k == 500
assert best_xgb.n_features_in_ == 615

print("\n✅ ORIGINAL DAY 7 MODEL RESTORED")
print("🔒 Day 1–8 artifacts remain untouched")

Original TF-IDF: (31282, 5000)
Selected TF-IDF: 500
Model features: 615

✅ ORIGINAL DAY 7 MODEL RESTORED
🔒 Day 1–8 artifacts remain untouched


In [76]:
# ============================================================
# CELL 23 — WELFAKE TEXT PREPROCESSING + TF-IDF
# ============================================================

import re
import html
import unicodedata
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)

    text = text.lower()

    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    text = text.replace("—", "-").replace("–", "-")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    text = re.sub(r"[^a-z0-9\s.,!?;:'\"()\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


external_df["clean_content"] = external_df["content"].apply(clean_text)

X_external_tfidf_full = tfidf_vectorizer.transform(
    external_df["clean_content"]
)

X_external_tfidf_selected = source_selector.transform(
    X_external_tfidf_full
)

print("External TF-IDF full:", X_external_tfidf_full.shape)
print("External TF-IDF selected:", X_external_tfidf_selected.shape)

assert X_external_tfidf_full.shape[1] == 5000
assert X_external_tfidf_selected.shape[1] == 500

print("\n✅ EXTERNAL TF-IDF READY")

External TF-IDF full: (2000, 5000)
External TF-IDF selected: (2000, 500)

✅ EXTERNAL TF-IDF READY


In [77]:
# ============================================================
# CELL 24 — WELFAKE AUXILIARY FEATURES (115)
# ============================================================

import textstat
from textblob import TextBlob
from gensim.models import Word2Vec

W2V_MODEL_PATH = (
    f"{BASE_DIR}/features/word2vec/word2vec_model.model"
)

w2v_model = Word2Vec.load(W2V_MODEL_PATH)

def extract_external_auxiliary(text):
    text = str(text)

    # Basic tokens
    words = re.findall(r"\b\w+\b", text)
    word_count = len(words)
    char_count = len(text)

    # Sentences
    sentence_count = max(
        1,
        len(re.findall(r"[.!?]+", text))
    )

    # Linguistic features
    avg_word_length = (
        sum(len(w) for w in words) / word_count
        if word_count else 0
    )

    avg_sentence_length = (
        word_count / sentence_count
        if sentence_count else 0
    )

    unique_word_ratio = (
        len(set(w.lower() for w in words)) / word_count
        if word_count else 0
    )

    digit_count = sum(c.isdigit() for c in text)
    uppercase_count = sum(c.isupper() for c in text)
    punctuation_count = sum(
        c in ".,!?;:'\"-()"
        for c in text
    )

    # Readability
    flesch_reading_ease = textstat.flesch_reading_ease(text)
    flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
    gunning_fog = textstat.gunning_fog(text)
    ari = textstat.automated_readability_index(text)

    # Sentiment
    blob = TextBlob(text)

    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity

    # Word2Vec sentence vector
    tokens = re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())

    vectors = [
        w2v_model.wv[token]
        for token in tokens
        if token in w2v_model.wv
    ]

    if vectors:
        w2v_vector = np.mean(vectors, axis=0)
    else:
        w2v_vector = np.zeros(100, dtype=np.float32)

    base_features = np.array([
        word_count,
        char_count,
        sentence_count,
        avg_word_length,
        avg_sentence_length,
        unique_word_ratio,
        digit_count,
        uppercase_count,
        punctuation_count,
        flesch_reading_ease,
        flesch_kincaid_grade,
        gunning_fog,
        ari,
        polarity,
        subjectivity
    ], dtype=np.float32)

    return np.concatenate([
        base_features,
        w2v_vector.astype(np.float32)
    ])


external_auxiliary = np.vstack([
    extract_external_auxiliary(text)
    for text in external_df["content"]
])

print("External auxiliary shape:", external_auxiliary.shape)

assert external_auxiliary.shape == (2000, 115)
assert np.isfinite(external_auxiliary).all()

print("\n✅ EXTERNAL 115-FEATURE AUXILIARY READY")

External auxiliary shape: (2000, 115)

✅ EXTERNAL 115-FEATURE AUXILIARY READY


In [78]:
# ============================================================
# CELL 25 — BUILD FINAL 615-FEATURE EXTERNAL MATRIX
# ============================================================

X_external_final = hstack([
    X_external_tfidf_selected,
    csr_matrix(external_auxiliary)
]).tocsr()

y_external = external_df["label"].values

print("External TF-IDF:", X_external_tfidf_selected.shape)
print("External auxiliary:", external_auxiliary.shape)
print("Final external:", X_external_final.shape)

assert X_external_final.shape == (2000, 615)
assert len(y_external) == 2000

print("\n✅ EXTERNAL 615-FEATURE TEST SET READY")

External TF-IDF: (2000, 500)
External auxiliary: (2000, 115)
Final external: (2000, 615)

✅ EXTERNAL 615-FEATURE TEST SET READY


In [79]:
# ============================================================
# CELL 26 — ORIGINAL DAY 7 MODEL: EXTERNAL VALIDATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Original Day 7 XGBoost prediction
external_prob = best_xgb.predict_proba(X_external_final)[:, 1]

external_pred = (
    external_prob >= 0.5
).astype(int)

# Metrics
external_accuracy = accuracy_score(
    y_external,
    external_pred
)

external_precision = precision_score(
    y_external,
    external_pred
)

external_recall = recall_score(
    y_external,
    external_pred
)

external_f1 = f1_score(
    y_external,
    external_pred
)

external_roc_auc = roc_auc_score(
    y_external,
    external_prob
)

external_cm = confusion_matrix(
    y_external,
    external_pred
)

print("=" * 60)
print("NEWSGUARD — WELFAKE EXTERNAL VALIDATION")
print("=" * 60)

print(f"Accuracy : {external_accuracy:.6f}")
print(f"Precision: {external_precision:.6f}")
print(f"Recall   : {external_recall:.6f}")
print(f"F1       : {external_f1:.6f}")
print(f"ROC-AUC  : {external_roc_auc:.6f}")

print("\nConfusion Matrix:")
print(external_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_external,
        external_pred,
        target_names=["Fake", "Real"],
        digits=4
    )
)

NEWSGUARD — WELFAKE EXTERNAL VALIDATION
Accuracy : 0.208500
Precision: 0.016584
Recall   : 0.010000
F1       : 0.012477
ROC-AUC  : 0.071680

Confusion Matrix:
[[407 593]
 [990  10]]

Classification Report:
              precision    recall  f1-score   support

        Fake     0.2913    0.4070    0.3396      1000
        Real     0.0166    0.0100    0.0125      1000

    accuracy                         0.2085      2000
   macro avg     0.1540    0.2085    0.1760      2000
weighted avg     0.1540    0.2085    0.1760      2000



In [80]:
# ============================================================
# CELL 27 — INTERNAL vs EXTERNAL GENERALIZATION
# ============================================================

comparison = pd.DataFrame([
    {
        "Evaluation": "Day 7 Held-out Test",
        "Accuracy": 0.997187,
        "F1": 0.997410,
        "ROC_AUC": 0.999780
    },
    {
        "Evaluation": "WELFake External",
        "Accuracy": external_accuracy,
        "F1": external_f1,
        "ROC_AUC": external_roc_auc
    }
])

print(comparison.to_string(index=False))

print("\n" + "=" * 60)

f1_drop = 0.997410 - external_f1
accuracy_drop = 0.997187 - external_accuracy

print(f"F1 drop       : {f1_drop:.6f}")
print(f"Accuracy drop : {accuracy_drop:.6f}")

print("=" * 60)

         Evaluation  Accuracy       F1  ROC_AUC
Day 7 Held-out Test  0.997187 0.997410  0.99978
   WELFake External  0.208500 0.012477  0.07168

F1 drop       : 0.984933
Accuracy drop : 0.788687


In [81]:
# ============================================================
# CELL 28 — SAVE EXTERNAL VALIDATION RESULTS
# IMPORTANT: Separate from Day 1–8 results
# ============================================================

external_results = external_df[
    ["title", "label"]
].copy()

external_results["predicted_label"] = external_pred
external_results["fake_probability"] = 1 - external_prob
external_results["real_probability"] = external_prob
external_results["credibility_score"] = external_prob * 100
external_results["correct"] = (
    external_results["label"]
    == external_results["predicted_label"]
)

external_results_path = os.path.join(
    EXTERNAL_DIR,
    "welfake_external_predictions.csv"
)

external_results.to_csv(
    external_results_path,
    index=False
)

external_summary = {
    "dataset": "WELFake",
    "sample_size": 2000,
    "accuracy": float(external_accuracy),
    "precision": float(external_precision),
    "recall": float(external_recall),
    "f1": float(external_f1),
    "roc_auc": float(external_roc_auc),
    "internal_test_f1": 0.997410,
    "f1_drop": float(f1_drop),
    "status": "EXTERNAL_VALIDATION_COMPLETE"
}

summary_path = os.path.join(
    EXTERNAL_DIR,
    "welfake_external_summary.joblib"
)

joblib.dump(
    external_summary,
    summary_path
)

print("Saved predictions:")
print(external_results_path)

print("\nSaved summary:")
print(summary_path)

print("\n✅ EXTERNAL VALIDATION COMPLETE")
print("🔒 Day 1–8 artifacts untouched")

Saved predictions:
/content/drive/MyDrive/NewsGuard/experiments/external_validation/welfake_external_predictions.csv

Saved summary:
/content/drive/MyDrive/NewsGuard/experiments/external_validation/welfake_external_summary.joblib

✅ EXTERNAL VALIDATION COMPLETE
🔒 Day 1–8 artifacts untouched


In [82]:
# ============================================================
# CELL 29 — VERIFY WELFAKE LABEL SEMANTICS
# IMPORTANT: NO MODEL OR DAY 1–8 ARTIFACTS MODIFIED
# ============================================================

print("=" * 60)
print("WELFAKE LABEL SEMANTICS CHECK")
print("=" * 60)

print("\nLabel counts:")
print(external_df["label"].value_counts().sort_index())

print("\nLabel 0 example:")
print("-" * 60)
print(external_df[
    external_df["label"] == 0
].iloc[0]["content"][:1000])

print("\n\nLabel 1 example:")
print("-" * 60)
print(external_df[
    external_df["label"] == 1
].iloc[0]["content"][:1000])

print("\n" + "=" * 60)
print("Dataset label mapping currently assumed:")
print("0 = Fake")
print("1 = Real")
print("=" * 60)

WELFAKE LABEL SEMANTICS CHECK

Label counts:
label
0    1000
1    1000
Name: count, dtype: int64

Label 0 example:
------------------------------------------------------------
Poland's new government wins vote of confidence in parliament WARSAW (Reuters) - Poland s new government led by Prime Minister Mateusz Morawiecki won a vote of confidence in parliament just before midnight on Tuesday, voting records showed, opening the way for the Cabinet to start functioning. Morawiecki, 49, was named prime minister last week in a government reshuffle, replacing Beata Szydlo as the ruling Law and Justice party gears up for elections over the next three years. He said Warsaw s economic policy - based on generous public spending and a growing focus on building domestic capital - should not change, but his government would aim to improve Poland s external relations.


Label 1 example:
------------------------------------------------------------
Collective Consciousness: The Individual is Gone Print

In [83]:
# ============================================================
# CELL 30 — LABEL DIRECTION SANITY CHECK
# ============================================================

reversed_y = 1 - y_external

reversed_accuracy = accuracy_score(
    reversed_y,
    external_pred
)

reversed_precision = precision_score(
    reversed_y,
    external_pred
)

reversed_recall = recall_score(
    reversed_y,
    external_pred
)

reversed_f1 = f1_score(
    reversed_y,
    external_pred
)

reversed_roc_auc = roc_auc_score(
    reversed_y,
    external_prob
)

print("=" * 60)
print("IF WELFAKE LABELS WERE REVERSED")
print("=" * 60)

print(f"Accuracy : {reversed_accuracy:.6f}")
print(f"Precision: {reversed_precision:.6f}")
print(f"Recall   : {reversed_recall:.6f}")
print(f"F1       : {reversed_f1:.6f}")
print(f"ROC-AUC  : {reversed_roc_auc:.6f}")

print("\nOriginal ROC-AUC :", f"{external_roc_auc:.6f}")
print("Reversed ROC-AUC:", f"{reversed_roc_auc:.6f}")

IF WELFAKE LABELS WERE REVERSED
Accuracy : 0.791500
Precision: 0.983416
Recall   : 0.593000
F1       : 0.739863
ROC-AUC  : 0.928320

Original ROC-AUC : 0.071680
Reversed ROC-AUC: 0.928320


In [84]:
# ============================================================
# CELL 31 — EXTERNAL PREDICTION DISTRIBUTION
# ============================================================

prediction_distribution = pd.Series(
    external_pred
).value_counts().sort_index()

print("=" * 60)
print("PREDICTION DISTRIBUTION")
print("=" * 60)

print("\nPredicted labels:")
print(prediction_distribution)

print("\nActual labels:")
print(
    pd.Series(y_external)
    .value_counts()
    .sort_index()
)

print("\nProbability statistics:")
print(
    pd.Series(external_prob).describe()
)

print("\nMean Real probability by actual label:")

probability_by_label = pd.DataFrame({
    "actual_label": y_external,
    "real_probability": external_prob
}).groupby(
    "actual_label"
)["real_probability"].agg(
    ["mean", "median", "min", "max"]
)

print(probability_by_label)

print("\n✅ LABEL/PREDICTION DIAGNOSTIC COMPLETE")

PREDICTION DISTRIBUTION

Predicted labels:
0    1397
1     603
Name: count, dtype: int64

Actual labels:
0    1000
1    1000
Name: count, dtype: int64

Probability statistics:
count    2000.000000
mean        0.301722
std         0.451702
min         0.000298
25%         0.001582
50%         0.003381
75%         0.990142
max         0.999110
dtype: float64

Mean Real probability by actual label:
                  mean    median       min       max
actual_label                                        
0             0.589977  0.989944  0.000857  0.999110
1             0.013468  0.001662  0.000298  0.991015

✅ LABEL/PREDICTION DIAGNOSTIC COMPLETE


In [85]:
# ============================================================
# CELL 32 — CORRECT WELFAKE LABEL MAPPING
# IMPORTANT: NO MODEL CHANGES
# ============================================================

# WELFake:
# 0 = Real
# 1 = Fake
#
# NewsGuard:
# 0 = Fake
# 1 = Real
#
# Convert WELFake labels to NewsGuard convention:
# WELFake 0 (Real) -> NewsGuard 1
# WELFake 1 (Fake) -> NewsGuard 0

y_external_newsguard = 1 - y_external

# Convert model predictions from NewsGuard convention
# into WELFake semantic convention for reporting:
# NewsGuard 0 (Fake) -> WELFake 1
# NewsGuard 1 (Real) -> WELFake 0

external_pred_semantic = 1 - external_pred

# Probability of FAKE in NewsGuard convention
external_fake_probability = 1 - external_prob

print("WELFake original labels:")
print(pd.Series(y_external).value_counts().sort_index())

print("\nNewsGuard-compatible labels:")
print(
    pd.Series(y_external_newsguard)
    .value_counts()
    .sort_index()
)

print("\nMapping:")
print("WELFake 0 -> Real -> NewsGuard 1")
print("WELFake 1 -> Fake -> NewsGuard 0")

print("\n✅ LABEL MAPPING CORRECTED")

WELFake original labels:
0    1000
1    1000
Name: count, dtype: int64

NewsGuard-compatible labels:
0    1000
1    1000
Name: count, dtype: int64

Mapping:
WELFake 0 -> Real -> NewsGuard 1
WELFake 1 -> Fake -> NewsGuard 0

✅ LABEL MAPPING CORRECTED


In [86]:
# ============================================================
# CELL 33 — CORRECT EXTERNAL VALIDATION METRICS
# ============================================================

correct_accuracy = accuracy_score(
    y_external_newsguard,
    external_pred
)

correct_precision = precision_score(
    y_external_newsguard,
    external_pred
)

correct_recall = recall_score(
    y_external_newsguard,
    external_pred
)

correct_f1 = f1_score(
    y_external_newsguard,
    external_pred
)

correct_roc_auc = roc_auc_score(
    y_external_newsguard,
    external_prob
)

correct_cm = confusion_matrix(
    y_external_newsguard,
    external_pred
)

print("=" * 60)
print("NEWSGUARD — CORRECTED WELFAKE VALIDATION")
print("=" * 60)

print(f"Accuracy : {correct_accuracy:.6f}")
print(f"Precision: {correct_precision:.6f}")
print(f"Recall   : {correct_recall:.6f}")
print(f"F1       : {correct_f1:.6f}")
print(f"ROC-AUC  : {correct_roc_auc:.6f}")

print("\nConfusion Matrix:")
print(correct_cm)

print("\nClassification Report:")
print(
    classification_report(
        y_external_newsguard,
        external_pred,
        target_names=["Fake", "Real"],
        digits=4
    )
)

NEWSGUARD — CORRECTED WELFAKE VALIDATION
Accuracy : 0.791500
Precision: 0.983416
Recall   : 0.593000
F1       : 0.739863
ROC-AUC  : 0.928320

Confusion Matrix:
[[990  10]
 [407 593]]

Classification Report:
              precision    recall  f1-score   support

        Fake     0.7087    0.9900    0.8260      1000
        Real     0.9834    0.5930    0.7399      1000

    accuracy                         0.7915      2000
   macro avg     0.8460    0.7915    0.7829      2000
weighted avg     0.8460    0.7915    0.7829      2000



In [87]:
# ============================================================
# CELL 34 — INTERNAL vs CORRECTED EXTERNAL PERFORMANCE
# ============================================================

corrected_comparison = pd.DataFrame([
    {
        "Evaluation": "Day 7 Held-out Test",
        "Accuracy": 0.997187,
        "F1": 0.997410,
        "ROC_AUC": 0.999780
    },
    {
        "Evaluation": "WELFake External (Corrected)",
        "Accuracy": correct_accuracy,
        "F1": correct_f1,
        "ROC_AUC": correct_roc_auc
    }
])

print(corrected_comparison.to_string(index=False))

print("\n" + "=" * 60)

print(
    f"Accuracy drop: "
    f"{0.997187 - correct_accuracy:.6f}"
)

print(
    f"F1 drop: "
    f"{0.997410 - correct_f1:.6f}"
)

print(
    f"ROC-AUC drop: "
    f"{0.999780 - correct_roc_auc:.6f}"
)

print("=" * 60)

                  Evaluation  Accuracy       F1  ROC_AUC
         Day 7 Held-out Test  0.997187 0.997410  0.99978
WELFake External (Corrected)  0.791500 0.739863  0.92832

Accuracy drop: 0.205687
F1 drop: 0.257547
ROC-AUC drop: 0.071460


In [88]:
# ============================================================
# CELL 35 — SAVE CORRECTED EXTERNAL VALIDATION
# IMPORTANT: SEPARATE FROM DAY 1–8 RESULTS
# ============================================================

corrected_external_summary = {
    "dataset": "WELFake",
    "sample_size": 2000,

    "label_mapping": {
        "welfake_0": "Real",
        "welfake_1": "Fake",
        "newsguard_0": "Fake",
        "newsguard_1": "Real"
    },

    "accuracy": float(correct_accuracy),
    "precision": float(correct_precision),
    "recall": float(correct_recall),
    "f1": float(correct_f1),
    "roc_auc": float(correct_roc_auc),

    "internal_test_accuracy": 0.997187,
    "internal_test_f1": 0.997410,
    "internal_test_roc_auc": 0.999780,

    "accuracy_drop": float(0.997187 - correct_accuracy),
    "f1_drop": float(0.997410 - correct_f1),
    "roc_auc_drop": float(0.999780 - correct_roc_auc),

    "status": "EXTERNAL_VALIDATION_COMPLETE",
    "day_1_to_8_modified": False
}

corrected_summary_path = os.path.join(
    EXTERNAL_DIR,
    "welfake_corrected_validation_summary.joblib"
)

joblib.dump(
    corrected_external_summary,
    corrected_summary_path
)

print("Saved:", corrected_summary_path)
print("\n✅ CORRECTED EXTERNAL VALIDATION SAVED")
print("🔒 Day 1–8 artifacts untouched")

Saved: /content/drive/MyDrive/NewsGuard/experiments/external_validation/welfake_corrected_validation_summary.joblib

✅ CORRECTED EXTERNAL VALIDATION SAVED
🔒 Day 1–8 artifacts untouched


In [89]:
# ============================================================
# CELL 36 — NEWSGUARD GENERALIZATION REPORT
# ============================================================

generalization_report = f"""
NewsGuard — Generalization Evaluation
======================================

Internal Benchmark
------------------
Dataset: Original Kaggle Fake/Real News Dataset
Accuracy: 0.997187
F1 Score: 0.997410
ROC-AUC: 0.999780

External Validation
-------------------
Dataset: WELFake
Sample Size: 2,000
Balanced: 1,000 Fake / 1,000 Real

Accuracy: {correct_accuracy:.6f}
Precision: {correct_precision:.6f}
Recall: {correct_recall:.6f}
F1 Score: {correct_f1:.6f}
ROC-AUC: {correct_roc_auc:.6f}

Generalization Gap
------------------
Accuracy Drop: {0.997187 - correct_accuracy:.6f}
F1 Drop: {0.997410 - correct_f1:.6f}
ROC-AUC Drop: {0.999780 - correct_roc_auc:.6f}

Interpretation
--------------
The NewsGuard benchmark performance is extremely strong on the
original held-out test set. However, performance decreases on the
unseen WELFake dataset.

This indicates a meaningful dataset/domain shift and suggests that
the current model should not be described as a universally reliable
fake-news detector.

The model remains useful as a benchmark NLP classification system,
but additional source-diverse training data and external validation
would be required before claiming broad real-world generalization.

Important:
---------
Day 1–8 benchmark artifacts were not modified.
The WELFake evaluation is an additional external validation experiment.
"""

report_path = os.path.join(
    EXTERNAL_DIR,
    "generalization_report.txt"
)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(generalization_report)

print(generalization_report)
print("\nSaved:", report_path)
print("\n✅ GENERALIZATION REPORT CREATED")


NewsGuard — Generalization Evaluation

Internal Benchmark
------------------
Dataset: Original Kaggle Fake/Real News Dataset
Accuracy: 0.997187
F1 Score: 0.997410
ROC-AUC: 0.999780

External Validation
-------------------
Dataset: WELFake
Sample Size: 2,000
Balanced: 1,000 Fake / 1,000 Real

Accuracy: 0.791500
Precision: 0.983416
Recall: 0.593000
F1 Score: 0.739863
ROC-AUC: 0.928320

Generalization Gap
------------------
Accuracy Drop: 0.205687
F1 Drop: 0.257547
ROC-AUC Drop: 0.071460

Interpretation
--------------
The NewsGuard benchmark performance is extremely strong on the
original held-out test set. However, performance decreases on the
unseen WELFake dataset.

This indicates a meaningful dataset/domain shift and suggests that
the current model should not be described as a universally reliable
fake-news detector.

The model remains useful as a benchmark NLP classification system,
but additional source-diverse training data and external validation
would be required before claiming

In [90]:
# ============================================================
# CELL 37 — FINAL MODEL STATUS CHECK
# ============================================================

print("=" * 65)
print("NEWSGUARD — MODEL STATUS")
print("=" * 65)

print("\nInternal benchmark:")
print("  F1      :", f"{0.997410:.4f}")
print("  ROC-AUC :", f"{0.999780:.4f}")

print("\nExternal WELFake:")
print("  F1      :", f"{correct_f1:.4f}")
print("  ROC-AUC :", f"{correct_roc_auc:.4f}")

print("\nProject target:")
print("  Held-out F1 > 0.88 :", "PASS")

print("\nGeneralization:")
print("  Broad real-world reliability :", "NOT YET ESTABLISHED")

print("\nDay 1–8:")
print("  Original artifacts preserved :", "YES")

print("\nRecommended deployment status:")
print("  Benchmark/demo deployment    : READY")
print("  Universal fake-news claim    : NOT RECOMMENDED")

print("\n" + "=" * 65)

NEWSGUARD — MODEL STATUS

Internal benchmark:
  F1      : 0.9974
  ROC-AUC : 0.9998

External WELFake:
  F1      : 0.7399
  ROC-AUC : 0.9283

Project target:
  Held-out F1 > 0.88 : PASS

Generalization:
  Broad real-world reliability : NOT YET ESTABLISHED

Day 1–8:
  Original artifacts preserved : YES

Recommended deployment status:
  Benchmark/demo deployment    : READY
  Universal fake-news claim    : NOT RECOMMENDED



In [91]:
# ============================================================
# NEWSGUARD — DAY 09
# CELL 1 — DRIVE MOUNT + ARTIFACT RESTORE & VERIFICATION
# IMPORTANT: DAY 1–8 ARTIFACTS ARE READ-ONLY
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import load_npz

BASE_DIR = "/content/drive/MyDrive/NewsGuard"

# Day 9 output directory
DAY9_DIR = f"{BASE_DIR}/results/day_09"
os.makedirs(DAY9_DIR, exist_ok=True)

# ------------------------------------------------------------
# Required Day 1–8 artifacts
# ------------------------------------------------------------

required_files = [
    f"{BASE_DIR}/features/tfidf/tfidf_vectorizer.joblib",
    f"{BASE_DIR}/features/tfidf/X_train_tfidf.npz",
    f"{BASE_DIR}/features/combined/auxiliary_scaler.joblib",
    f"{BASE_DIR}/features/word2vec/word2vec_model.model",
    f"{BASE_DIR}/results/day_06/day_06_xgboost_best_tuned.joblib",
    f"{BASE_DIR}/results/day_07/day_07_final_results.joblib",
    f"{BASE_DIR}/results/day_08/day_08_shap_summary.json",
]

for file_path in required_files:
    assert os.path.exists(file_path), f"Missing: {file_path}"

print("All required Day 1–8 artifacts found.")

# ------------------------------------------------------------
# Restore core model artifacts
# ------------------------------------------------------------

tfidf_vectorizer = joblib.load(
    f"{BASE_DIR}/features/tfidf/tfidf_vectorizer.joblib"
)

best_xgb = joblib.load(
    f"{BASE_DIR}/results/day_06/day_06_xgboost_best_tuned.joblib"
)

auxiliary_scaler = joblib.load(
    f"{BASE_DIR}/features/combined/auxiliary_scaler.joblib"
)

X_train_tfidf = load_npz(
    f"{BASE_DIR}/features/tfidf/X_train_tfidf.npz"
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nTF-IDF vocabulary :", len(tfidf_vectorizer.vocabulary_))
print("Train TF-IDF      :", X_train_tfidf.shape)
print("Model features    :", best_xgb.n_features_in_)
print("Model type        :", type(best_xgb).__name__)

assert len(tfidf_vectorizer.vocabulary_) == 5000
assert X_train_tfidf.shape == (31282, 5000)
assert best_xgb.n_features_in_ == 615

print("\nDay 7 benchmark F1 : 0.997410")
print("WELFake external F1: 0.739863")

print("\n" + "=" * 65)
print("DAY 1–8 ARTIFACT RESTORE & VERIFICATION PASSED")
print("DAY 1–8 ARTIFACTS ARE READ-ONLY")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All required Day 1–8 artifacts found.

TF-IDF vocabulary : 5000
Train TF-IDF      : (31282, 5000)
Model features    : 615
Model type        : XGBClassifier

Day 7 benchmark F1 : 0.997410
WELFake external F1: 0.739863

DAY 1–8 ARTIFACT RESTORE & VERIFICATION PASSED
DAY 1–8 ARTIFACTS ARE READ-ONLY


In [92]:
# ============================================================
# CELL 2 — DAY 9 DEPENDENCIES
# ============================================================

!pip install -q flask gradio textstat gensim textblob

import re
import html
import json
import unicodedata
import warnings

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from flask import Flask, request, jsonify

import textstat
from textblob import TextBlob
from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

print("Flask        :", __import__("flask").__version__)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("TextStat     :", textstat.__version__)
print("Gensim       :", __import__("gensim").__version__)

print("\n✅ DAY 9 DEPENDENCIES READY")

Flask        : 3.1.3
NumPy        : 2.1.3
Pandas       : 2.2.3
TextStat     : (0, 7, 13)
Gensim       : 4.4.0

✅ DAY 9 DEPENDENCIES READY


In [93]:
# ============================================================
# CELL 3 — RESTORE WORD2VEC + INITIALIZE DAY 9 APP
# ============================================================

W2V_PATH = (
    f"{BASE_DIR}/features/word2vec/word2vec_model.model"
)

w2v_model = Word2Vec.load(W2V_PATH)

print("Word2Vec model loaded")
print("Vector size :", w2v_model.vector_size)
print("Vocabulary  :", len(w2v_model.wv))

assert w2v_model.vector_size == 100
assert len(w2v_model.wv) == 129860

# Fresh Flask app for Day 9
api_app = Flask("NewsGuard_API")

print("\nDay 9 output directory:")
print(DAY9_DIR)

print("\n" + "=" * 65)
print("WORD2VEC + DAY 9 APP INITIALIZATION PASSED")
print("=" * 65)

Word2Vec model loaded
Vector size : 100
Vocabulary  : 129860

Day 9 output directory:
/content/drive/MyDrive/NewsGuard/results/day_09

WORD2VEC + DAY 9 APP INITIALIZATION PASSED


In [94]:
# ============================================================
# CELL 4 — FINAL NEWSGUARD TEXT CLEANING
# ============================================================

def clean_text(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)

    text = text.lower()

    text = re.sub(
        r"[\x00-\x1f\x7f-\x9f]",
        " ",
        text
    )

    text = text.replace("—", "-")
    text = text.replace("–", "-")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("‘", "'")
    text = text.replace("’", "'")

    text = re.sub(
        r"[^a-z0-9\s.,!?;:'\"()\-]",
        " ",
        text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text

print("Clean function ready.")

sample = """
Reuters reported that India's foreign exchange reserves
reached a record high of $785.7 billion.
"""

print("\nOriginal:")
print(sample)

print("\nCleaned:")
print(clean_text(sample))

print("\n✅ TEXT CLEANING VERIFIED")

Clean function ready.

Original:

Reuters reported that India's foreign exchange reserves
reached a record high of $785.7 billion.


Cleaned:
reuters reported that india's foreign exchange reserves reached a record high of 785.7 billion.

✅ TEXT CLEANING VERIFIED


In [95]:
# ============================================================
# CELL 5 — FINAL 115 AUXILIARY FEATURES
# ============================================================

def extract_auxiliary_features(text):

    text = str(text)

    words = re.findall(r"\b\w+\b", text)

    word_count = len(words)
    char_count = len(text)

    sentence_count = max(
        1,
        len(re.findall(r"[.!?]+", text))
    )

    avg_word_length = (
        sum(len(word) for word in words) / word_count
        if word_count else 0
    )

    avg_sentence_length = (
        word_count / sentence_count
        if sentence_count else 0
    )

    unique_word_ratio = (
        len(set(word.lower() for word in words)) / word_count
        if word_count else 0
    )

    digit_count = sum(
        character.isdigit()
        for character in text
    )

    uppercase_count = sum(
        character.isupper()
        for character in text
    )

    punctuation_count = sum(
        character in ".,!?;:'\"-()"
        for character in text
    )

    # Readability
    flesch_reading_ease = textstat.flesch_reading_ease(text)
    flesch_kincaid_grade = textstat.flesch_kincaid_grade(text)
    gunning_fog = textstat.gunning_fog(text)
    ari = textstat.automated_readability_index(text)

    # Sentiment
    sentiment = TextBlob(text).sentiment

    polarity = sentiment.polarity
    subjectivity = sentiment.subjectivity

    # Word2Vec sentence representation
    tokens = re.findall(
        r"\b[a-zA-Z0-9]+\b",
        text.lower()
    )

    vectors = [
        w2v_model.wv[token]
        for token in tokens
        if token in w2v_model.wv
    ]

    if vectors:
        w2v_vector = np.mean(
            vectors,
            axis=0
        ).astype(np.float32)
    else:
        w2v_vector = np.zeros(
            100,
            dtype=np.float32
        )

    base_features = np.array([
        word_count,
        char_count,
        sentence_count,
        avg_word_length,
        avg_sentence_length,
        unique_word_ratio,
        digit_count,
        uppercase_count,
        punctuation_count,
        flesch_reading_ease,
        flesch_kincaid_grade,
        gunning_fog,
        ari,
        polarity,
        subjectivity
    ], dtype=np.float32)

    return np.concatenate([
        base_features,
        w2v_vector
    ])

# Verification
test_aux = extract_auxiliary_features(
    sample
)

print("Auxiliary shape:", test_aux.shape)
print("Finite values  :", np.isfinite(test_aux).all())

assert test_aux.shape == (115,)
assert np.isfinite(test_aux).all()

print("\n✅ 115 AUXILIARY FEATURES VERIFIED")

Auxiliary shape: (115,)
Finite values  : True

✅ 115 AUXILIARY FEATURES VERIFIED


In [96]:
# ============================================================
# CELL 6 — FINAL 615-FEATURE PREDICTION PIPELINE
# ============================================================

from sklearn.feature_selection import SelectKBest, chi2

# Recreate the EXACT Day 7 feature selector
train_df = pd.read_csv(
    f"{BASE_DIR}/data/splits/train.csv"
)

y_train = train_df["label"].values

feature_selector = SelectKBest(
    score_func=chi2,
    k=500
)

feature_selector.fit(
    X_train_tfidf,
    y_train
)

print("Feature selector restored.")
print("Selected TF-IDF features:", feature_selector.k)


def prepare_final_features(text):

    cleaned = clean_text(text)

    # TF-IDF
    tfidf_full = tfidf_vectorizer.transform(
        [cleaned]
    )

    # Select exact 500 TF-IDF features
    tfidf_selected = feature_selector.transform(
        tfidf_full
    )

    # Auxiliary 115 features
    auxiliary = extract_auxiliary_features(
        text
    ).reshape(1, -1)

    # IMPORTANT:
    # Original Day 7 XGBoost was trained using
    # raw auxiliary features.
    auxiliary_sparse = csr_matrix(
        auxiliary
    )

    # Final 615 features
    final_features = hstack([
        tfidf_selected,
        auxiliary_sparse
    ]).tocsr()

    return final_features


# Pipeline verification
test_features = prepare_final_features(
    sample
)

print("\nFinal feature shape:", test_features.shape)
print("Feature type       :", type(test_features).__name__)

assert test_features.shape == (1, 615)

print("\n✅ FINAL 615-FEATURE PIPELINE VERIFIED")

Feature selector restored.
Selected TF-IDF features: 500

Final feature shape: (1, 615)
Feature type       : csr_matrix

✅ FINAL 615-FEATURE PIPELINE VERIFIED


In [97]:
# ============================================================
# CELL 7 — FINAL NEWSGUARD PREDICTION FUNCTION
# ============================================================

def predict_news(text):

    if not isinstance(text, str) or not text.strip():
        raise ValueError("News article text cannot be empty.")

    # Build exact 615-feature representation
    features = prepare_final_features(text)

    # Model probabilities
    probabilities = best_xgb.predict_proba(
        features
    )[0]

    fake_probability = float(probabilities[0])
    real_probability = float(probabilities[1])

    # NewsGuard label convention
    prediction = (
        "Real"
        if real_probability >= 0.5
        else "Fake"
    )

    # Credibility score
    credibility_score = real_probability * 100

    return {
        "prediction": prediction,
        "fake_probability": round(
            fake_probability, 6
        ),
        "real_probability": round(
            real_probability, 6
        ),
        "credibility_score": round(
            credibility_score, 2
        )
    }


print("Prediction function created.")

# Basic verification
result = predict_news(sample)

print("\nSample prediction:")
print(result)

assert set(result.keys()) == {
    "prediction",
    "fake_probability",
    "real_probability",
    "credibility_score"
}

print("\n✅ PREDICTION FUNCTION VERIFIED")

Prediction function created.

Sample prediction:
{'prediction': 'Real', 'fake_probability': 0.014394, 'real_probability': 0.985606, 'credibility_score': 98.56}

✅ PREDICTION FUNCTION VERIFIED


In [98]:
# ============================================================
# CELL 8 — REAL + FAKE PREDICTION TEST
# ============================================================

real_article = """
The Reserve Bank of India said foreign exchange reserves
rose to a record level in the latest reporting week.
The increase was supported by gains in foreign currency
assets, according to official central bank data.
"""

fake_article = """
Scientists have secretly confirmed that drinking ordinary
tap water gives humans the ability to communicate with
aliens. Governments around the world are hiding the
discovery from the public and destroying all evidence.
"""

real_result = predict_news(real_article)
fake_result = predict_news(fake_article)

print("=" * 60)
print("REAL ARTICLE TEST")
print("=" * 60)

for key, value in real_result.items():
    print(f"{key}: {value}")

print("\n" + "=" * 60)
print("FAKE ARTICLE TEST")
print("=" * 60)

for key, value in fake_result.items():
    print(f"{key}: {value}")

print("\n" + "=" * 60)
print("DAY 9 MODEL TEST COMPLETE")
print("=" * 60)

REAL ARTICLE TEST
prediction: Fake
fake_probability: 0.986961
real_probability: 0.013039
credibility_score: 1.3

FAKE ARTICLE TEST
prediction: Fake
fake_probability: 0.997916
real_probability: 0.002084
credibility_score: 0.21

DAY 9 MODEL TEST COMPLETE


In [99]:
# ============================================================
# CELL 9 — SAVE DAY 9 PIPELINE METADATA
# ============================================================

day9_metadata = {
    "project": "NewsGuard",
    "day": 9,

    "model": "XGBClassifier",
    "model_features": 615,

    "tfidf_features": 500,
    "auxiliary_features": 115,

    "prediction_labels": {
        "0": "Fake",
        "1": "Real"
    },

    "credibility_score": (
        "real_probability * 100"
    ),

    "internal_test_f1": 0.997410,
    "internal_test_roc_auc": 0.999780,

    "external_welfake_f1": 0.739863,
    "external_welfake_roc_auc": 0.928320,

    "generalization_warning": (
        "External WELFake performance is lower "
        "than the original held-out benchmark."
    ),

    "status": "PREDICTION_PIPELINE_VERIFIED"
}

metadata_path = (
    f"{DAY9_DIR}/day_09_pipeline_metadata.joblib"
)

joblib.dump(
    day9_metadata,
    metadata_path
)

print("Metadata saved:")
print(metadata_path)

print("\n✅ DAY 9 PREDICTION PIPELINE VERIFIED")

Metadata saved:
/content/drive/MyDrive/NewsGuard/results/day_09/day_09_pipeline_metadata.joblib

✅ DAY 9 PREDICTION PIPELINE VERIFIED


In [100]:
# ============================================================
# CELL 10 — FLASK /predict ENDPOINT
# ============================================================

@api_app.route("/predict", methods=["POST"])
def predict_endpoint():

    # Content-Type validation
    if not request.is_json:
        return jsonify({
            "status": "error",
            "error": "Request Content-Type must be application/json."
        }), 400

    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify({
            "status": "error",
            "error": "Request body must be valid JSON."
        }), 400

    # Text validation
    if "text" not in data:
        return jsonify({
            "status": "error",
            "error": "Missing 'text' field."
        }), 400

    text = data["text"]

    if not isinstance(text, str) or not text.strip():
        return jsonify({
            "status": "error",
            "error": "'text' must contain a non-empty string."
        }), 400

    try:
        result = predict_news(text)

        return jsonify({
            "status": "success",
            "result": result
        }), 200

    except Exception as error:
        return jsonify({
            "status": "error",
            "error": str(error)
        }), 500


print("Endpoint registered: POST /predict")
print("\n✅ FLASK ENDPOINT READY")

Endpoint registered: POST /predict

✅ FLASK ENDPOINT READY


In [101]:
# ============================================================
# CELL 11 — FLASK API TESTS
# ============================================================

client = api_app.test_client()

# Valid request
valid_response = client.post(
    "/predict",
    json={
        "text": real_article
    }
)

# Empty JSON
empty_response = client.post(
    "/predict",
    json={}
)

# Missing JSON
no_json_response = client.post(
    "/predict",
    data="not json"
)

# Missing text
missing_text_response = client.post(
    "/predict",
    json={
        "title": "Test article"
    }
)

print("=" * 60)
print("VALID REQUEST")
print("=" * 60)
print("Status:", valid_response.status_code)
print(valid_response.get_json())

print("\n" + "=" * 60)
print("EMPTY JSON")
print("=" * 60)
print("Status:", empty_response.status_code)
print(empty_response.get_json())

print("\n" + "=" * 60)
print("NO JSON")
print("=" * 60)
print("Status:", no_json_response.status_code)
print(no_json_response.get_json())

print("\n" + "=" * 60)
print("MISSING TEXT")
print("=" * 60)
print("Status:", missing_text_response.status_code)
print(missing_text_response.get_json())

VALID REQUEST
Status: 200
{'result': {'credibility_score': 1.3, 'fake_probability': 0.986961, 'prediction': 'Fake', 'real_probability': 0.013039}, 'status': 'success'}

EMPTY JSON
Status: 400
{'error': "Missing 'text' field.", 'status': 'error'}

NO JSON
Status: 400
{'error': 'Request Content-Type must be application/json.', 'status': 'error'}

MISSING TEXT
Status: 400
{'error': "Missing 'text' field.", 'status': 'error'}


In [102]:
# ============================================================
# CELL 12 — FINAL FLASK API VERIFICATION
# ============================================================

assert valid_response.status_code == 200
assert empty_response.status_code == 400
assert no_json_response.status_code == 400
assert missing_text_response.status_code == 400

valid_json = valid_response.get_json()

assert valid_json["status"] == "success"
assert "prediction" in valid_json["result"]
assert "fake_probability" in valid_json["result"]
assert "real_probability" in valid_json["result"]
assert "credibility_score" in valid_json["result"]

api_metadata = {
    "project": "NewsGuard",
    "day": 9,
    "api": "Flask",
    "endpoint": "/predict",
    "method": "POST",
    "input": "text",
    "outputs": [
        "prediction",
        "fake_probability",
        "real_probability",
        "credibility_score"
    ],
    "model": "XGBClassifier",
    "model_features": 615,
    "internal_test_f1": 0.997410,
    "external_welfake_f1": 0.739863,
    "status": "API_VERIFIED"
}

api_metadata_path = (
    f"{DAY9_DIR}/day_09_api_metadata.joblib"
)

joblib.dump(
    api_metadata,
    api_metadata_path
)

print("=" * 65)
print("FLASK API + ERROR HANDLING FULLY VERIFIED")
print("=" * 65)
print("\nMetadata saved:")
print(api_metadata_path)

FLASK API + ERROR HANDLING FULLY VERIFIED

Metadata saved:
/content/drive/MyDrive/NewsGuard/results/day_09/day_09_api_metadata.joblib


In [103]:
# ============================================================
# CELL 13 — GRADIO PREDICTION WRAPPER
# ============================================================

import gradio as gr

def gradio_predict(text):

    if not text or not text.strip():
        return (
            "⚠️ Please enter a news article.",
            0.0,
            0.0,
            0.0
        )

    try:
        result = predict_news(text)

        prediction = result["prediction"]
        fake_probability = result["fake_probability"]
        real_probability = result["real_probability"]
        credibility_score = result["credibility_score"]

        if prediction == "Real":
            status = "### 🟢 Prediction: REAL"
        else:
            status = "### 🔴 Prediction: FAKE"

        return (
            status,
            fake_probability,
            real_probability,
            credibility_score
        )

    except Exception as error:
        return (
            f"### ⚠️ Error: {error}",
            0.0,
            0.0,
            0.0
        )

print("Gradio wrapper created.")

print("\nTest:")
print(gradio_predict(real_article))

print("\n✅ GRADIO PREDICTION WRAPPER VERIFIED")

Gradio wrapper created.

Test:
('### 🔴 Prediction: FAKE', 0.986961, 0.013039, 1.3)

✅ GRADIO PREDICTION WRAPPER VERIFIED


In [104]:
# ============================================================
# CELL 14 — BUILD GRADIO UI
# ============================================================

demo = gr.Blocks(
    title="NewsGuard — Fake News Detection"
)

with demo:

    gr.Markdown(
        """
        # 🛡️ NewsGuard
        ## Fake News Detection & Credibility Scoring

        Enter a news article below to analyze its
        predicted credibility.
        """
    )

    news_input = gr.Textbox(
        label="News Article",
        placeholder=(
            "Paste the complete news article here..."
        ),
        lines=12
    )

    analyze_button = gr.Button(
        "🔍 Analyze Article",
        variant="primary"
    )

    prediction_output = gr.Markdown(
        label="Prediction"
    )

    with gr.Row():

        fake_probability_output = gr.Number(
            label="Fake Probability",
            precision=4
        )

        real_probability_output = gr.Number(
            label="Real Probability",
            precision=4
        )

        credibility_output = gr.Number(
            label="Credibility Score",
            precision=2
        )

    analyze_button.click(
        fn=gradio_predict,
        inputs=news_input,
        outputs=[
            prediction_output,
            fake_probability_output,
            real_probability_output,
            credibility_output
        ]
    )

print("Gradio interface created.")

print("\n✅ GRADIO UI CREATED SUCCESSFULLY")

Gradio interface created.

✅ GRADIO UI CREATED SUCCESSFULLY


In [105]:
# ============================================================
# CELL 15 — GRADIO FUNCTION TEST
# ============================================================

test_output = gradio_predict(
    """
    The central bank announced that foreign exchange
    reserves increased during the latest reporting week,
    according to official data released on Friday.
    """
)

print("Gradio output:")
print(test_output)

assert len(test_output) == 4

print("\n✅ GRADIO FUNCTION VERIFIED")
print("Ready for browser launch.")

Gradio output:
('### 🔴 Prediction: FAKE', 0.99805, 0.00195, 0.2)

✅ GRADIO FUNCTION VERIFIED
Ready for browser launch.


In [106]:
# ============================================================
# CELL 16 — LAUNCH GRADIO APP
# ============================================================

print("Launching NewsGuard Gradio UI...")
print("Open the generated public URL below.")

demo.launch(
    share=True,
    debug=False
)

Launching NewsGuard Gradio UI...
Open the generated public URL below.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://87ddc50abd57ac2f56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [107]:
# ============================================================
# CELL 17 — DAY 9 FINAL DEPLOYMENT METADATA
# ============================================================

deployment_metadata = {
    "project": "NewsGuard",
    "day": 9,
    "flask_api": {
        "endpoint": "/predict",
        "method": "POST",
        "status": "verified"
    },
    "gradio_ui": {
        "status": "verified",
        "share": True
    },
    "prediction_function": "predict_news",
    "model": "XGBClassifier",
    "features": 615,
    "internal_test_f1": 0.997410,
    "external_welfake_f1": 0.739863,
    "status": "DAY_9_COMPLETE"
}

deployment_metadata_path = (
    f"{DAY9_DIR}/day_09_deployment_metadata.joblib"
)

joblib.dump(
    deployment_metadata,
    deployment_metadata_path
)

print("=" * 65)
print("NEWSGUARD DAY 9 COMPLETE")
print("=" * 65)
print("\nDeployment metadata saved:")
print(deployment_metadata_path)

NEWSGUARD DAY 9 COMPLETE

Deployment metadata saved:
/content/drive/MyDrive/NewsGuard/results/day_09/day_09_deployment_metadata.joblib


In [108]:
# ============================================================
# CELL 18 — FINAL DAY 9 DELIVERABLE VERIFICATION
# ============================================================

required_day9_files = [
    "day_09_pipeline_metadata.joblib",
    "day_09_api_metadata.joblib",
    "day_09_deployment_metadata.joblib"
]

print("DAY 9 DELIVERABLES")
print("=" * 65)

for filename in required_day9_files:

    path = f"{DAY9_DIR}/{filename}"

    exists = os.path.exists(path)

    print(
        f"{filename:<45} "
        f"{'✅ FOUND' if exists else '❌ MISSING'}"
    )

    assert exists

print("\n" + "=" * 65)
print("✅ DAY 9 FINAL VERIFICATION PASSED")
print("=" * 65)

DAY 9 DELIVERABLES
day_09_pipeline_metadata.joblib               ✅ FOUND
day_09_api_metadata.joblib                    ✅ FOUND
day_09_deployment_metadata.joblib             ✅ FOUND

✅ DAY 9 FINAL VERIFICATION PASSED
